# Milestone 2, Task 4: False-negative analysis

## Settings

In [ ]:
MODE = "demo"
INPUT_PATH = ""
SAMPLE_PER_CATEGORY = 3
RANDOM_SEED = 42
DOWNLOAD_RESULTS = False

## Validate inputs and calculate missed labels

In [ ]:
import json
import hashlib
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def validate_bundle(bundle):
    if bundle.get("schema_version") != 1:
        raise ValueError("Unsupported input schema.")
    if not isinstance(bundle.get("synthetic"), bool):
        raise ValueError("Input must declare whether it is synthetic.")
    labels = bundle["labels"]
    chunks = pd.DataFrame(bundle["chunks"])
    truth = np.asarray(bundle["y_true"])
    if not labels or len(labels) != len(set(labels)):
        raise ValueError("Label names must be nonempty and unique.")
    required = {"contract_title", "chunk_id", "chunk_text"}
    if not required.issubset(chunks.columns) or chunks.empty:
        raise ValueError("Missing chunk metadata or empty dataset.")
    if chunks[list(required)].isna().any().any():
        raise ValueError("Chunk metadata contains nulls.")
    if chunks.duplicated(["contract_title", "chunk_id"]).any():
        raise ValueError("Duplicate contract/chunk identifiers.")
    if truth.shape != (len(chunks), len(labels)) or not np.isin(truth, [0, 1]).all():
        raise ValueError("Invalid true labels or alignment dimensions.")
    if not bundle["models"]:
        raise ValueError("No model predictions included.")
    for model, values in bundle["models"].items():
        pred = np.asarray(values["predictions"])
        if pred.shape != truth.shape or not np.isin(pred, [0, 1]).all():
            raise ValueError(f"Invalid predictions for {model}.")
        if "probabilities" in values:
            p = np.asarray(values["probabilities"], dtype=float)
            if p.shape != truth.shape or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
                raise ValueError(f"Invalid probabilities for {model}.")
    return chunks, truth.astype(int), labels

def divide(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return np.divide(a, b, out=np.full_like(a, np.nan), where=b != 0)

def analyze(bundle):
    chunks, truth, labels = validate_bundle(bundle)
    results = {}
    for model, values in bundle["models"].items():
        pred = np.asarray(values["predictions"])
        proba = np.asarray(values["probabilities"]) if "probabilities" in values else None
        tp = ((truth == 1) & (pred == 1)).sum(axis=0)
        fn_mask = (truth == 1) & (pred == 0)
        fn = fn_mask.sum(axis=0)
        summary = pd.DataFrame({"category": labels, "support": truth.sum(axis=0),
            "predicted_positive": pred.sum(axis=0), "TP": tp, "FN": fn,
            "recall": divide(tp, tp+fn), "miss_rate": divide(fn, tp+fn)})
        reviews = {}
        for kind, mask, owner in [("FN", fn_mask, "Aarnav")]:
            rows, cols = np.where(mask)
            review = chunks.iloc[rows].copy().reset_index(drop=True)
            review.insert(0, "validation_row", rows)
            review.insert(0, "model", model)
            review["error_type"] = kind
            review["category"] = [labels[j] for j in cols]
            review["true_label"] = truth[rows, cols]
            review["predicted_label"] = pred[rows, cols]
            review["probability"] = proba[rows, cols] if proba is not None else np.nan
            review["assigned_reviewer"] = owner
            for field in ["review_status", "possible_reason", "supporting_evidence", "suggested_improvement"]:
                review[field] = ""
            reviews[kind] = review
        overall = {"model": model, "chunks": len(chunks), "labels": len(labels),
            "TP": int(tp.sum()), "FN": int(fn.sum()),
            "predicted_positive_pairs": int(pred.sum()),
            "micro_recall": float(divide(tp.sum(), (tp+fn).sum())),
            "overall_miss_rate": float(divide(fn.sum(), (tp+fn).sum()))}
        results[model] = {"summary": summary, "reviews": reviews, "overall": overall}
    return results


## Test with synthetic data

In [ ]:
def demo_bundle():
    return {"schema_version": 1, "synthetic": True, "run_description": "Synthetic test only",
        "labels": ["Demo A", "Demo B", "Demo C"],
        "chunks": [{"contract_title": "Demo contract", "chunk_id": str(i),
                    "chunk_text": f"Invented test chunk {i}."} for i in range(3)],
        "y_true": [[1,0,0], [1,1,0], [0,1,0]],
        "models": {"demo": {"predictions": [[0,1,0], [1,0,0], [0,1,0]],
                            "probabilities": [[.2,.8,.1],[.8,.3,.1],[.1,.9,.1]]}}}

import copy
d = demo_bundle()
r = analyze(d)["demo"]
assert r["overall"]["FN"] == 2
assert r["overall"]["TP"] == 2
assert np.isclose(r["overall"]["micro_recall"], .5)
assert np.isclose(r["overall"]["overall_miss_rate"], .5)
assert r["reviews"]["FN"]["validation_row"].tolist() == [0,1]
assert r["reviews"]["FN"]["category"].tolist() == ["Demo A", "Demo B"]
perfect = copy.deepcopy(d)
perfect["models"]["demo"] = {"predictions": perfect["y_true"]}
assert analyze(perfect)["demo"]["reviews"]["FN"].empty
assert analyze(perfect)["demo"]["overall"]["micro_recall"] == 1
negative = copy.deepcopy(d)
negative["models"]["demo"] = {"predictions": [[0,0,0]]*3}
assert analyze(negative)["demo"]["overall"]["FN"] == 4
assert analyze(negative)["demo"]["overall"]["micro_recall"] == 0
assert np.isnan(r["summary"].loc[r["summary"]["category"] == "Demo C", "recall"].iloc[0])
bad = copy.deepcopy(d)
bad["models"]["demo"]["predictions"] = [[0,1]]
try:
    analyze(bad)
except ValueError:
    pass
else:
    raise AssertionError("Invalid input was accepted.")
print("All synthetic validation checks passed. These are not project findings.")


## Load validation predictions

In [ ]:
if MODE == "demo":
    bundle = demo_bundle()
elif MODE == "real":
    if INPUT_PATH:
        bundle = json.loads(Path(INPUT_PATH).read_text())
    else:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one task4_inputs.json file.")
        bundle = json.loads(next(iter(uploaded.values())))
    if bundle.get("synthetic") is not False:
        raise ValueError("Real mode cannot use a synthetic bundle.")
else:
    raise ValueError("MODE must be demo or real.")
results = analyze(bundle)
print("SYNTHETIC DEMO ONLY" if bundle["synthetic"] else "REAL VALIDATION DATA")
print("Run:", bundle["run_description"])
display(pd.DataFrame([r["overall"] for r in results.values()]))


## Summarize and sample false negatives

In [ ]:
review_samples = {}
for model, result in results.items():
    print("MODEL:", model)
    display(result["summary"].sort_values("FN", ascending=False))
    for kind in ["FN"]:
        table = result["reviews"][kind]
        parts = [group.sample(n=min(SAMPLE_PER_CATEGORY, len(group)), random_state=RANDOM_SEED)
                 for _, group in table.groupby("category", sort=True)]
        sample = pd.concat(parts, ignore_index=True) if parts else table.copy()
        review_samples[(model, kind)] = sample
        print(kind, "total pairs:", len(table), "| review sample:", len(sample))
        display(sample.head(12))
    if result["overall"]["predicted_positive_pairs"] == 0:
        print("No positive labels were predicted. Check model outputs, training and threshold; this alone does not establish the cause.")
    top = result["summary"].nlargest(10, "FN")
    ax = top.set_index("category")[["FN"]].plot.barh(figsize=(9,5))
    ax.set_title(("DEMO — " if bundle["synthetic"] else "") + model + ": largest missed-label counts")
    ax.set_xlabel("Chunk–label pairs")
    plt.tight_layout()
    plt.show()


## Export results

In [ ]:
prefix = "DEMO_task4_FN_results" if bundle["synthetic"] else "task4_FN_results"
out = Path(prefix)
out.mkdir(exist_ok=True)
written = []
def save_csv(frame, name):
    path = out / name
    frame.to_csv(path, index=False)
    written.append(path)

save_csv(pd.DataFrame([r["overall"] for r in results.values()]), "overall_metrics.csv")
for idx, (model, result) in enumerate(results.items(), start=1):
    # Numeric prefixes avoid treating arbitrary model names as filesystem paths.
    tag = f"model_{idx}"
    save_csv(result["summary"], f"{tag}_category_summary.csv")
    for kind in ["FN"]:
        save_csv(result["reviews"][kind], f"{tag}_{kind}_all.csv")
        save_csv(review_samples[(model, kind)], f"{tag}_{kind}_review_sample.csv")
provenance = {"synthetic": bundle["synthetic"], "run_description": bundle["run_description"],
    "model_file_mapping": {f"model_{i}": m for i,m in enumerate(results,1)},
    "input_sha256": hashlib.sha256(json.dumps(bundle, sort_keys=True).encode()).hexdigest(),
    "sample_per_category": SAMPLE_PER_CATEGORY, "random_seed": RANDOM_SEED,
    "manual_review_status": "Pending; blank fields do not represent completed review.",
    "undefined_metrics": "NaN/blank when denominator is zero"}
p = out / "provenance.json"
p.write_text(json.dumps(provenance, indent=2))
written.append(p)
archive = Path(prefix + ".zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in written:
        z.write(p, arcname=p.name)
print("Saved:", archive, "| manual review remains pending")
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive))


## Export predictions from the model runtime

In [ ]:
EXPORT_INPUTS = False

if EXPORT_INPUTS:
    import json
    import numpy as np
    from datetime import datetime, timezone
    from google.colab import files

    MODEL_SELECTION = "both"
    RUN_DESCRIPTION = ""
    ALIGNMENT_CONFIRMED = False

    if not ALIGNMENT_CONFIRMED or not RUN_DESCRIPTION.strip():
        raise ValueError("Confirm alignment and fill RUN_DESCRIPTION before export.")
    choices = {"tfidf": ("y_val_pred", "y_val_proba"),
               "bert": ("val_predictions", "val_probabilities")}
    if MODEL_SELECTION not in ("tfidf", "bert", "both"):
        raise ValueError("Choose tfidf, bert, or both.")
    selected = list(choices) if MODEL_SELECTION == "both" else [MODEL_SELECTION]
    required = ["val_chunks_df", "y_val", "mlb"] + [choices[m][0] for m in selected]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise ValueError("Missing runtime variables: " + ", ".join(missing) +
                         ". Run upstream cells or choose the available model.")

    chunks = val_chunks_df.reset_index(drop=True)
    truth = np.asarray(y_val)
    labels = [str(x) for x in mlb.classes_]
    if truth.shape != (len(chunks), len(labels)) or not np.isin(truth, [0, 1]).all():
        raise ValueError("Invalid label matrix or row/column alignment.")
    if not {"contract_title", "chunk_id", "chunk_text", "categories"}.issubset(chunks.columns):
        raise ValueError("Expected chunk metadata and categories are missing.")
    if not np.array_equal(mlb.transform(chunks["categories"]), truth):
        raise ValueError("Chunk annotations do not match y_val in this order.")
    records = []
    for _, row in chunks.iterrows():
        records.append({"contract_title": str(row["contract_title"]),
                        "chunk_id": str(row["chunk_id"]),
                        "chunk_text": str(row["chunk_text"])})
    models = {}
    for name in selected:
        pred_name, proba_name = choices[name]
        pred = np.asarray(globals()[pred_name])
        if pred.shape != truth.shape or not np.isin(pred, [0, 1]).all():
            raise ValueError(f"Invalid {name} predictions.")
        entry = {"predictions": pred.astype(int).tolist()}
        if proba_name in globals():
            proba = np.asarray(globals()[proba_name], dtype=float)
            if proba.shape != truth.shape or not np.isfinite(proba).all() or ((proba < 0) | (proba > 1)).any():
                raise ValueError(f"Invalid {name} probabilities.")
            entry["probabilities"] = proba.tolist()
        models[name] = entry
    bundle = {"schema_version": 1, "synthetic": False,
              "run_description": RUN_DESCRIPTION,
              "exported_at": datetime.now(timezone.utc).isoformat(),
              "labels": labels, "chunks": records,
              "y_true": truth.astype(int).tolist(), "models": models}
    with open("task4_inputs.json", "w") as f:
        json.dump(bundle, f, allow_nan=False)
    files.download("task4_inputs.json")
    print("Exported:", selected, "| chunks:", len(chunks), "| labels:", len(labels))